[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/27_vit_patch.ipynb)

# 🟠 中等：ViT 补丁嵌入

实现 Vision Transformer (ViT) 的 **补丁嵌入** 层。

### 函数签名
```python
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, C, H, W)
        # 返回: (B, num_patches, embed_dim)
```

### 算法
1. 将图像重塑为非重叠补丁：`(B, C, H, W)` → `(B, N, C*P*P)`
2. 投影每个补丁：`nn.Linear(C*P*P, embed_dim)`
3. `num_patches = (img_size // patch_size) ** 2`

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✏️ 在此实现你的代码

class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()
        pass  # self.num_patches, self.proj

    def forward(self, x):
        pass  # 重塑为补丁，投影

使用 kernal=patch_size, stride=patch_size, output_channels=embed_dim 的 conv2D 对图片进行投影，将图片投影成 (B, (img_size // patch_size) ** 2, embed_dim) 的特征矩阵

In [ ]:
from torch import Tensor

class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        """
        Vision Transformer 的补丁嵌入层
        
        Args:
            img_size: 输入图像尺寸（假设为正方形，H=W=img_size）
            patch_size: 每个补丁的尺寸（假设为正方形），每个补丁投影至 embed_dim
            in_channels: 输入图像的通道数
            embed_dim: 嵌入维度
        """
        super(PatchEmbedding, self).__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.embed_dim = embed_dim
        
        # 计算补丁数量
        self.num_patches = (img_size // patch_size) ** 2
        
        # 将图像分割为补丁并展平的卷积层
        # 使用卷积实现：kernel_size=patch_size, stride=patch_size
        # 输出形状: (B, embed_dim, num_patches_h, num_patches_w)
        self.proj = nn.Conv2d(
            in_channels, 
            embed_dim, 
            kernel_size=patch_size, 
            stride=patch_size
        )
        
    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x: 输入图像 (B, C, H, W)
        Returns:
            Tensor: 补丁嵌入 (B, num_patches, embed_dim)
        """
        B, C, H, W = x.shape
        
        # 验证输入尺寸
        assert H == self.img_size and W == self.img_size, \
            f"输入尺寸应为 ({self.img_size}, {self.img_size})，但得到 ({H}, {W})"
        assert C == self.in_channels, \
            f"通道数应为 {self.in_channels}，但得到 {C}"
        
        # 使用卷积投影： (B, embed_dim, num_patches_h, num_patches_w)
        x = self.proj(x)
        
        # 展平空间维度： (B, embed_dim, num_patches)
        x = x.flatten(2)
        
        # 转置为 (B, num_patches, embed_dim)
        x = x.transpose(1, 2)
        
        return x


In [ ]:
# 🧪 调试
pe = PatchEmbedding(32, 8, 3, 64)
x = torch.randn(2, 3, 32, 32)
print('输出:', pe(x).shape)
print('补丁数:', pe.num_patches)

In [ ]:
# ✅ 提交
from torch_judge import check
check('vit_patch')